# Running `llama.cpp` as a server with a quantized model and multi-token-prediction

Now, we will try yet another software [`llama.cpp`](https://github.com/ggml-org/llama.cpp).
Originally, `llama.cpp` was conceived for CPU-only usage, but it has gained broad
hardware support and will also run with CUDA and many other accelerators.

`llama.cpp` is written in (you guess it) `C++` and has an OpenAI compatible interface.

`llama.cpp` now also interfaces with the Hugging Face ecosystem and can download
models automatically.

We will use the OpenAI client:

In [ ]:
from openai import OpenAI

An API key can be added, but we don't need it here

In [ ]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8090/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

Now we use a larger model: `llama-server   -hf unsloth/Qwen3.6-27B-MTP-GGUF:UD-Q4_K_XL   --spec-type draft-mtp --spec-draft-n-max 4   -ngl 99 -fa on --host 0.0.0.0 --port 8090`

Compared to `vllm`, the startup time is again much faster. Let's see
if we can also gain additional speed with generation:

In [ ]:
model = "Qwen3.6-27B-MTP-GGUF:UD-Q4_K_XL"

In [ ]:
%%time
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "Explain O'Reilly online learning!" } ]
)

Note the speed of the generation (in the terminal window). This is also very
fast, even if it cannot always reach the speed of `exl3`.

In [ ]:
from IPython.display import display, Markdown
display(Markdown(completion.choices[0].message.content))

The `responses` API is also `llama.cpp`. Let's generate `python` code, which supposedly can also
be created by the draft model:

In [ ]:
%%time
response = client.responses.create(model=model, 
                                   input="Write a quicksort in Python")

In [ ]:
display(Markdown(response.output_text))

Now, take a look at the GPU memory usage.

In [ ]:
!nvidia-smi

`llama.cpp` now uses more memory. Of course the model is also larger, but
still the speed is very good!